# AutoDrive Gym — GRPO Training on Colab

**OpenEnv Hackathon Submission** | [HF Space](https://huggingface.co/spaces/openenv-community/autodrive) | [GitHub](https://github.com/YOUR_GH_USER/autodrive_openenv)

Train an autonomous driving agent for **Indian road conditions** using **GRPO** with HF TRL — importing all training logic directly from the `autodrive_env` package.

| Component | Detail |
|-----------|--------|
| **Environment** | AutoDrive Gym — 30+ Indian road hazard scenarios |
| **Training** | This Colab notebook (T4 / A10G GPU) |
| **Model** | `Qwen/Qwen3-0.6B` or `Qwen/Qwen3-1.7B` + LoRA |
| **Framework** | HF TRL v0.29+ GRPO with vLLM backend |
| **Reward signals** | Dense per-step + phase-order bonus + resolution scaling |

> ⚠️ **Set Runtime → Change runtime type → T4 GPU** before running any cells.


## 1. Install Dependencies

Install `autodrive_env` (includes training utils, environment, models) and TRL with vLLM backend.


In [ ]:
# Clone the repo (skip if already present)
import os
if not os.path.exists("autodrive_openenv"):
    !git clone https://github.com/YOUR_GH_USER/autodrive_openenv.git
%cd autodrive_openenv

# Install autodrive_env package + GRPO extras (torch, trl, vllm, peft, transformers, datasets)
!pip install -q -e ".[grpo]"
print("✅ Installation complete.")


## 2. Configuration

Set the model, training hyperparameters, and (optionally) HuggingFace Hub repo. Add `HF_TOKEN` to Colab Secrets (key icon in sidebar).


In [ ]:
import os

# --- HuggingFace token ---
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except (ImportError, KeyError, ModuleNotFoundError):
    if "HF_TOKEN" not in os.environ:
        print("⚠️  HF_TOKEN not found. Set it in Colab Secrets or as an env var.")
        print("   Qwen3-0.6B is public — token only needed to push a checkpoint to Hub.")

# --- Model ---
MODEL_ID = "Qwen/Qwen3-0.6B"   # 0.6B = T4 (16 GB) | 1.7B = A10G (24 GB) | 4B = A100
HUB_REPO = ""                   # e.g. "your-hf-username/autodrive-agent"  (leave "" to skip)

# --- Training ---
NUM_EPISODES    = 50   # total driving episodes
NUM_GENERATIONS = 8    # GRPO rollouts per prompt (≥ 8 for stable advantage estimates)
MAX_TURNS       = 20   # max steps per episode
LORA_R          = 8    # LoRA rank: 8 = 8 GB / 16 = 16 GB / 32 = 40 GB+
LEARNING_RATE   = 2e-6
MAX_NEW_TOKENS  = 256  # JSON action is ~60 tokens — keep short
REPORT_TO       = "none"   # "wandb" | "tensorboard" | "none"

print(f"Model       : {MODEL_ID}")
print(f"Episodes    : {NUM_EPISODES}")
print(f"Generations : {NUM_GENERATIONS}")
print(f"LoRA rank   : {LORA_R}")
print(f"Hub repo    : {HUB_REPO or '(not set — will not push)'}")


## 3. Smoke Test — Verify Environment

Reset the gym and run one step to confirm the environment imports and responds correctly before using GPU time.


In [ ]:
import sys
sys.path.insert(0, ".")

from server.autodrive_gym_environment import AutoDriveGymEnvironment
from models import AutoDriveAction

env = AutoDriveGymEnvironment()
obs = env.reset()

print("✅ Environment ready!")
print(f"\nScenario   : {obs.scenario_type}")
print(f"Stage      : {obs.scenario_stage}")
print(f"Hazard     : {obs.hazard_type} at {obs.hazard_distance:.1f}m")
print(f"Situation  : {(obs.command_output or '')[:200]}")

# One step to verify the full loop works
result = env.step(AutoDriveAction(action="brake", value=0.8))
print(f"\nStep reward : {result.reward:.3f}")
print(f"Done        : {result.done}")
print("Smoke test passed ✅")


## 4. Import Training Utilities from Package

All training logic (system prompt, rollout function, observation formatter, reward functions, TRL patch) is imported directly from `train_grpo` — the same code used for production training. No duplication.


In [ ]:
import logging
from datasets import Dataset
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer
from trl.experimental.openenv import generate_rollout_completions

# Import ALL training utilities from train_grpo — single source of truth
from train_grpo import (
    SYSTEM_PROMPT,
    VALID_ACTIONS,
    format_observation,
    parse_action,
    rollout_once,
    reward_total,
    reward_success,
    patch_trl_vllm_compat,
)

# Apply TRL / vLLM compatibility patch (needed on some TRL versions)
patch_trl_vllm_compat()

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

print("System prompt (first 300 chars):")
print(SYSTEM_PROMPT[:300])
print("...\n")
print(f"Imported: rollout_once, format_observation, parse_action, reward_total, reward_success ✅")


## 5. Training Setup

Tokenizer, environment, dataset, reward logger, and rollout function — using the utilities imported from the package.


In [ ]:
import csv
from datetime import datetime
from pathlib import Path

# ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── Environment (local — no HTTP, direct Python import) ───────────────────────
from server.autodrive_gym_environment import AutoDriveGymEnvironment
train_env = AutoDriveGymEnvironment()

# ── Dataset (one entry = one episode prompt) ──────────────────────────────────
dataset = Dataset.from_dict(
    {"prompt": ["Navigate Indian road conditions safely."] * NUM_EPISODES}
)

# ── Output directory ──────────────────────────────────────────────────────────
ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = Path(f"outputs/autodrive-grpo-{MODEL_ID.replace('/','--')}-{ts}")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output dir : {output_dir}")

# ── Reward CSV logger ─────────────────────────────────────────────────────────
reward_log_path = output_dir / "reward_log.csv"
episode_counter = [0]
all_rewards, all_successes = [], []

with open(reward_log_path, "w", newline="") as f:
    csv.writer(f).writerow(["episode", "total_reward", "success", "steps", "timestamp"])

def log_episode(total_r: float, success: bool, steps: int) -> None:
    episode_counter[0] += 1
    ep = episode_counter[0]
    all_rewards.append(total_r)
    all_successes.append(int(success))
    with open(reward_log_path, "a", newline="") as f:
        csv.writer(f).writerow([ep, total_r, int(success), steps, datetime.now().isoformat()])
    last10 = all_rewards[-10:]
    logger.info(
        "Episode %3d: reward=%.3f  success=%s  steps=%d | mean=%.3f  mean(10)=%.3f",
        ep, total_r, "✅" if success else "❌", steps,
        sum(all_rewards) / len(all_rewards),
        sum(last10) / len(last10),
    )

# ── Rollout function (calls rollout_once from train_grpo) ─────────────────────
def rollout_func(prompts, trainer):
    results = {k: [] for k in ["prompt_ids", "completion_ids", "logprobs",
                                "total_reward", "success"]}
    for _ in prompts:
        ep = rollout_once(trainer, train_env, tokenizer, MAX_TURNS)
        for k in results:
            results[k].append(ep[k])
        log_episode(ep["total_reward"], ep["success"], ep["steps"])
    return results

print("Training setup complete ✅")


## 6. GRPO Config + Trainer

Configure GRPOTrainer with LoRA, vLLM colocate, and DAPO loss (same as production `train_grpo.py`).


In [ ]:
grpo_config = GRPOConfig(
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.5,     # leaves headroom for LoRA backward pass
    output_dir=str(output_dir),
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=2,
    max_grad_norm=1.0,
    gradient_accumulation_steps=8,
    per_device_train_batch_size=1,
    generation_batch_size=NUM_GENERATIONS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_NEW_TOKENS,
    logging_steps=1,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    temperature=1.0,                     # T=1.0 maximises action diversity for GRPO
    report_to=REPORT_TO,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    push_to_hub=bool(HUB_REPO),
    hub_model_id=HUB_REPO if HUB_REPO else None,
    hub_strategy="every_save",
    loss_type="dapo",                    # DAPO: improved GRPO from DeepSeek
    mask_truncated_completions=True,
    beta=0.01,
)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_R * 2,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

trainer = GRPOTrainer(
    model=MODEL_ID,
    processing_class=tokenizer,
    reward_funcs=[reward_total, reward_success],
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
    peft_config=peft_config,
)

print("GRPOTrainer initialized ✅")
print(f"  model     : {MODEL_ID}")
print(f"  episodes  : {NUM_EPISODES}")
print(f"  gen×ep    : {NUM_GENERATIONS} rollouts per step")
print(f"  output    : {output_dir}")


## 7. Train!

Launch GRPO training. Each episode: reset scenario → agent generates JSON action → environment steps → reward computed → GRPO gradient update across 8 rollouts.


In [ ]:
print("Starting GRPO training...")
print(f"  Model       : {MODEL_ID}")
print(f"  Episodes    : {NUM_EPISODES}")
print(f"  Generations : {NUM_GENERATIONS}")
print()

try:
    trainer.train()
finally:
    pass  # env stays alive for reward curve cell below

trainer.save_model(str(output_dir))
print(f"\n✅ Model saved to {output_dir}")


## 8. Reward Curves

Visualize training progress — raw per-episode reward, rolling average, and success rate.


In [ ]:
import csv
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

episodes, rewards, successes = [], [], []
with open(reward_log_path) as f:
    for row in csv.DictReader(f):
        episodes.append(int(row["episode"]))
        rewards.append(float(row["total_reward"]))
        successes.append(int(row["success"]))

window = min(10, len(episodes))
rolling = [
    sum(rewards[max(0, i - window + 1): i + 1]) / min(i + 1, window)
    for i in range(len(rewards))
]
success_rate = [
    sum(successes[max(0, i - window + 1): i + 1]) / min(i + 1, window) * 100
    for i in range(len(successes))
]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                                facecolor="#F8F9FA",
                                gridspec_kw={"height_ratios": [3, 1]})
fig.suptitle(f"AutoDrive Gym — GRPO Training  ({MODEL_ID})", fontsize=14, fontweight="bold")

ax1.plot(episodes, rewards, alpha=0.25, color="#2196F3", linewidth=1,
         marker="o", markersize=3, label="Per-episode reward")
ax1.plot(episodes, rolling, color="#2196F3", linewidth=2.5,
         label=f"Rolling avg (window={window})")
ax1.axhline(0, color="#aaa", linestyle="--", linewidth=0.8)
ax1.set_ylabel("Episode reward")
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

ax2.bar(episodes, success_rate, color="#4CAF50", alpha=0.7)
ax2.set_ylabel("Success %")
ax2.set_xlabel("Episode")
ax2.set_ylim(0, 110)
ax2.grid(True, alpha=0.3)

out_png = output_dir / "reward_curve.png"
plt.tight_layout()
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.show()

print(f"Reward curve saved to {out_png}")
print(f"Final rolling avg  : {rolling[-1]:.3f}")
print(f"Final success rate : {success_rate[-1]:.1f}%")
print(f"Best episode reward: {max(rewards):.3f}")


## 9. Push to Hub (Optional)

Upload the fine-tuned LoRA adapter to HuggingFace Hub.


In [ ]:
# Uncomment and set HUB_REPO in Cell 2 to push your checkpoint:
# trainer.push_to_hub(repo_id=HUB_REPO)
# print(f"Model pushed to https://huggingface.co/{HUB_REPO}")
print("Set HUB_REPO in Cell 2 and uncomment the lines above to push.")
